# F16 — Baseline DL (ResNet-1D) sobre Colab T4

Consume los artefactos F15 (shards en Drive, manifiesto/splits/stats en
git) y entrena el baseline según `docs/plan_dl_zzu.md` §3 (F16):

1. Humo CPU (~1.000 ventanas, 2 épocas): pérdida decrece, F1 macro val
   se calcula → `F16 HUMO VERDE`.
2. Entrenamiento completo en T4 (checkpoint por época en Drive,
   Colab preempta). **Gate: F1 macro val > 0,518** (batir a RF); si no,
   máximo 1 iteración de diagnóstico antes de escalar al equipo.

No agrega torch a las dependencias del repo: Colab lo trae. Política
13-oct: cómputo pesado en Colab; el repo solo recibe artefactos ligeros
(historial.json, gate_f16.txt; mejor.pt queda en Drive para F17/F18).

## 1. Setup SSH (opcional, igual que 05_dl_colab)

In [ ]:
%%bash
apt-get -qq update >/dev/null && apt-get -qq install -y openssh-server >/dev/null
mkdir -p /run/sshd /root/.ssh && chmod 700 /root/.ssh
sed -i 's/#\?Port .*/Port 22/' /etc/ssh/sshd_config
nohup /usr/sbin/sshd >/dev/null 2>&1 &
echo "sshd arriba"

In [ ]:
PUBKEY = "ssh-ed25519 AAAA... tu@pc"
TS_AUTHKEY = "tskey-auth-..."

open("/root/.ssh/authorized_keys", "w").write(PUBKEY + "\n")
import os; os.chmod("/root/.ssh/authorized_keys", 0o600)
!curl -fsSL https://tailscale.com/install.sh | sh
# Colab no tiene systemd: levantar tailscaled en userspace antes de `up`
!tailscaled --tun=userspace-networking --socks5=localhost:1055 &>/dev/null & sleep 2
!tailscale up --hostname colab-f16 --authkey="$TS_AUTHKEY" --ssh=false
!tailscale ip -4

## 2. Repo + shards desde Drive

In [ ]:
REPO = "https://codeberg.org/norlek/" \
       "Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG.git"
RAMA = "etapa2-corpus-combinado"

!test -d Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG || git clone --depth 1 "$REPO" -b "$RAMA"
%cd Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
!git fetch origin "$RAMA" --depth 1
!git checkout -B "$RAMA" FETCH_HEAD
!git log --oneline -1
!pip install -q pandas numpy  # torch ya viene en Colab

from google.colab import drive
drive.mount('/content/drive')
# shards (6,6 GB) desde Drive al disco local del runtime: lectura
# memmap sobre Drive sería ~10x más lenta por epoch
import os, glob, time
if len(glob.glob('results/dl/shards/shard_*.npy')) < 7:
    os.makedirs('results/dl/shards', exist_ok=True)
    t0 = time.time()
    !cp /content/drive/MyDrive/arritmias_dl/shards/*.npy results/dl/shards/
    print(f'shards copiados en {time.time()-t0:.0f}s')
!ls -lh results/dl/shards | tail -3
assert len(glob.glob('results/dl/shards/shard_*.npy')) == 7

## 3. Humo CPU (~1.000 ventanas, 2 épocas) — correr ANTES de pedir T4

Aceptación: `F16 HUMO VERDE` (pérdida decrece + F1 macro val calculable).

In [ ]:
!python scripts/entrenar_baseline_dl.py --humo

## 4. Entrenamiento completo (runtime T4)

Checkpoints (`mejor.pt`, `ultimo.pt`) e `historial.json` directos a Drive
(reanudable tras preempt: relanzar esta celda y cargar `ultimo.pt` si hace
falta; F17 traerá `--reanudar`). Gate al final: `gate_f16.txt`.

In [ ]:
SALIDA = '/content/drive/MyDrive/arritmias_dl/baseline'
!mkdir -p "$SALIDA"
!python scripts/entrenar_baseline_dl.py --epocas 30 --batch 64 --lr 3e-4 --salida "$SALIDA"
!cat "$SALIDA/gate_f16.txt"

## 5. Persistencia de artefactos ligeros al repo

`mejor.pt`/`ultimo.pt` quedan en Drive (F17/F18). Al repo solo van
`historial.json` + `gate_f16.txt`. El push requiere credenciales: si
falla (Colab no las tiene), copia los dos archivos al repo local y
haz commit+push desde el PC — mismo flujo que F15.

In [ ]:
SALIDA = '/content/drive/MyDrive/arritmias_dl/baseline'
import os, shutil
os.makedirs('results/dl/baseline', exist_ok=True)
for f in ('historial.json', 'gate_f16.txt'):
    shutil.copy(f'{SALIDA}/{f}', f'results/dl/baseline/{f}')
!git config user.email "gustavo.castro14159@protonmail.com" && git config user.name "norlek"
!git add results/dl/baseline/historial.json results/dl/baseline/gate_f16.txt
!git commit -m "F16: baseline ResNet-1D en Colab T4 (gate en gate_f16.txt)"
!git push origin etapa2-corpus-combinado || echo 'PUSH FALLÓ: haz commit+push desde el PC con los archivos en Drive'